In [2]:
import pandas as pd
import numpy as np
import joblib

### wczytanie danych
df = joblib.load('df_with_all_clusters.joblib')
embeddings_full = np.load('embeddings_full.npy')
print("Obiekty wczytane.")

### sprawdzenie że czy jest tyle samo embeddingów co wierszy w DataFrame
assert len(df) == len(embeddings_full), "Niezgodna liczba dokumentów i embeddingów!"

Obiekty wczytane.


In [4]:
### potrzebna lista unikalnych ID dla każdego dokumentu (najlepiej indeksy DataFrame)
### ChromaDB wymaga ID jako stringów!!!
doc_ids = [str(i) for i in df.index]

print(f"Przygotowano dane dla {len(doc_ids)} dokumentów.")

Przygotowano dane dla 850367 dokumentów.


In [6]:
### Inicjalizacja ChromaDB i dodanie danych

import chromadb
import time
from tqdm import tqdm


#### zarządzanie bazą danych w folderze 'chroma_db'
client = chromadb.PersistentClient(path="chroma_db")


#### nazwa kolekcji (odpowiednik tabeli w SQL)
collection_name = "covid_articles"


try:
    client.delete_collection(name=collection_name)
    print(f"Usunięto istniejącą kolekcję: {collection_name}")
except:
    pass


#### nowa kolekcja
print(f"Tworzenie nowej kolekcji: {collection_name}")
collection = client.create_collection(name=collection_name)


#### dodawanie danych do kolekcji w partiach
batch_size = 5000
num_batches = int(np.ceil(len(doc_ids) / batch_size))


print(f"\nDodawanie {len(doc_ids)} dokumentów do bazy w {num_batches} partiach...")
start_time = time.time()


for i in tqdm(range(num_batches)):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(doc_ids))
    
    #### dane dla bieżącej partii
    batch_ids = doc_ids[start_idx:end_idx]
    #### ChromaDB oczekuje listy wektorów a nie tablicy NumPy
    batch_embeddings = embeddings_full[start_idx:end_idx].tolist() 
    
    #### dodanie partii do kolekcji
    collection.add(
        embeddings=batch_embeddings,
        ids=batch_ids
    )


end_time = time.time()
print(f"\nDane zostały dodane do bazy ChromaDB w {(end_time - start_time) / 60:.2f} minut.")

Tworzenie nowej kolekcji: covid_articles

Dodawanie 850367 dokumentów do bazy w 171 partiach...


100%|██████████| 171/171 [05:26<00:00,  1.91s/it]


Dane zostały dodane do bazy ChromaDB w 5.44 minut.


In [2]:
### Wyszukiwanie semantyczne w bazie ChromaDB

import chromadb
from sentence_transformers import SentenceTransformer
import time
import pandas as pd
import numpy as np
import joblib


### wczytanie danych
df = joblib.load('df_with_all_clusters.joblib')
embeddings_full = np.load('embeddings_full.npy')
print("Obiekty wczytane.")


#### wczytanie modelu SentenceTransformer (ten sam co do utworzenia embeddingów ; wektoryzacja ze SPECTER)
#### model 'allenai-specter' specjalnie zaprojektowany do reprezentacji artykułów naukowych
model = SentenceTransformer('allenai-specter', device='cuda')
print("Model załadowany.")


#### inicjalizacja klienta ChromaDB (istniejąca baza)
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection(name="covid_articles")

#### Definicja funkcji wyszukującej
def search_similar_articles(query_text, n_results=5):
    """
    Wyszukuje w bazie ChromaDB artykuły najbardziej podobne semantycznie do podanego zapytania.
    """
    print(f"\nWyszukiwanie dla zapytania: '{query_text}'")
    start_time = time.time()
    
    ## konwersja zapytania na embedding
    query_embedding = model.encode(query_text, convert_to_tensor=True, device='cuda')
    ## ChromaDB oczekuje listy embeddingów
    query_embeddings_list = [query_embedding.cpu().numpy().tolist()]
    
    ## zapytanie do bazy ChromaDB
    ##### 'query_embeddings': wektor(y), którego podobieństwo szukamy
    ##### 'n_results': ile wyników chcemy otrzymać
    results = collection.query(
        query_embeddings=query_embeddings_list,
        n_results=n_results
    )
    
    end_time = time.time()
    print(f"Wyszukiwanie zakończone w {(end_time - start_time):.3f} sekund.")
    
    ## wyniki
    if results and results.get('ids') and results['ids'][0]:
        found_ids = [int(id_str) for id_str in results['ids'][0]] ## ID z powrotem na liczby
        distances = results['distances'][0] if results.get('distances') else ["N/A"] * len(found_ids)
        
        print(f"\nZnaleziono {len(found_ids)} podobnych artykułów:")
        ## pobranie tytułów z DataFrame
        found_titles = df.loc[found_ids]['title']
        
        for i, (doc_id, title) in enumerate(found_titles.items()):
            print(f"{i+1}. (ID: {doc_id}, Odległość: {distances[i]:.4f}) {title}")
    else:
        print("Nie znaleziono żadnych wyników.")



Obiekty wczytane.
Model załadowany.



## Testowanie funkcji wyszukującej


In [4]:
### Test a)

search_similar_articles("effectiveness of masks in children", n_results=10)


Wyszukiwanie dla zapytania: 'effectiveness of masks in children'
Wyszukiwanie zakończone w 1.796 sekund.

Znaleziono 10 podobnych artykułów:
1. (ID: 1026403, Odległość: 138.1113) [An Evaluation of the Validity and Reliability of the Face Mask Use Scale's Korean Version among Community-Dwelling Adults]
2. (ID: 301800, Odległość: 139.5796) Increased Mask Use and Fewer Gatherings Associated with Lower SARS-CoV-2 Seropositivity Among Young School-Age Children (preprint)
3. (ID: 410903, Odległość: 141.8592) Adverse Effects of COVID-19 and Face Masks: A Systematic Review
4. (ID: 472165, Odległość: 141.8713) Environmental Hygiene for COVID-19: It’s All About the Mask
5. (ID: 586245, Odległość: 142.8659) Maternal perception of masking in children as a preventive strategy for COVID-19 in Nigeria: A multicentre study
6. (ID: 128528, Odległość: 142.9478) Effectiveness of community face mask use on COVID-19 epidemiological trends and patterns in Italy: evidence from a "translational" study.
7. (I

In [6]:
### Test b)

search_similar_articles("long covid symptoms neurological", n_results=10)


Wyszukiwanie dla zapytania: 'long covid symptoms neurological'
Wyszukiwanie zakończone w 0.000 sekund.

Znaleziono 10 podobnych artykułów:
1. (ID: 599067, Odległość: 67.1883) Covid-19. Neurological manifestations
2. (ID: 659853, Odległość: 67.1883) COVID-19 Neurological Manifestations
3. (ID: 16143, Odległość: 69.1596) Long COVID: understanding the neurological effects
4. (ID: 872417, Odległość: 71.2044) COVID-19 Associated Neurological Manifestations
5. (ID: 507637, Odległość: 72.3947) Potential neurological symptoms of COVID-19
6. (ID: 833886, Odległość: 77.5988) Post–COVID-19 neurological symptoms
7. (ID: 318576, Odległość: 84.0207) Neurological issues in COVID-19, summarized in verse
8. (ID: 957374, Odległość: 84.0207) Neurological Issues in COVID-19, Summarized in Verse
9. (ID: 74338, Odległość: 84.7842) LONG COVID
10. (ID: 419547, Odległość: 84.7842) Long Covid


In [8]:
### Test c)

search_similar_articles("economic impact of pandemic on tourism", n_results=10)


Wyszukiwanie dla zapytania: 'economic impact of pandemic on tourism'
Wyszukiwanie zakończone w 0.007 sekund.

Znaleziono 10 podobnych artykułów:
1. (ID: 204010, Odległość: 81.7626) Economic losses in tourism during the COVID-19 pandemic. The case of Sorrento
2. (ID: 661964, Odległość: 81.7658) Does the size of the tourism sector influence the economic policy response to the COVID-19 pandemic?
3. (ID: 427588, Odległość: 89.0234) Restructuring Tourism Industry in the after month of Corona (COVID-19) Pandemic: Sri Lankan Perspective (preprint)
4. (ID: 1027783, Odległość: 92.3449) Economic impacts of COVID-19 on the tourism sector in Tanzania
5. (ID: 356446, Odległość: 93.2457) Impact of the COVID-19 pandemic on rural tourism in Czechia Preliminary considerations
6. (ID: 247707, Odległość: 97.8413) Is it better to support tourism after pandemic regionally or globally? Case of European Union from the socio-economic perspective
7. (ID: 443089, Odległość: 98.3993) The impact of quarantine du